# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [9]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [11]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [12]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [13]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N1: Bayes Naive + TF-IDF con priors por defecto

Los priors son la probabilidad que el modelo le asigna a cada clase antes de ver el texto.

Naive Bayes clasifica calculando, para cada clase, la probabilidad de que un texto pertenezca a ella. Esa probabilidad tiene dos partes que se multiplican:

P(clase | texto) = P(texto | clase) × P(clase)

P(texto | clase) es la probabilidad de ver esas palabras dado que la reseña es, por ejemplo, negativa.

P(clase) es el prior, qué tan probable es esa clase en general, sin importar el texto.

Con priors por defecto (class_prior=None), los aprende del data: negativa 40%, neutra 20%, positiva 40%. Entonces antes de leer una sola palabra, el modelo ya cree que una reseña tiene el doble de chances de ser negativa o positiva que neutra.

In [14]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("nb",    MultinomialNB(alpha=1.0))])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_val)
evaluate("nb_tfidf_v1", y_val, y_pred, hyperparams={"alpha": 1.0, "vectorizer": "TF-IDF", "class_prior": "default"})

# Reentrenar en train completo y generar submission
pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(pipe, "nb_tfidf_v1")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_nb_tfidf_v1.csv");


=== nb_tfidf_v1 ===
Hiperparámetros: {'alpha': 1.0, 'vectorizer': 'TF-IDF', 'class_prior': 'default'}

F1-macro:  0.5753
Precision: 0.6544
Recall:    0.6089
Accuracy:  0.7130

              precision    recall  f1-score   support

    negativa     0.7090    0.8750    0.7833      4080
      neutra     0.5233    0.0882    0.1510      2040
    positiva     0.7308    0.8635    0.7916      4080

    accuracy                         0.7130     10200
   macro avg     0.6544    0.6089    0.5753     10200
weighted avg     0.6806    0.7130    0.6602     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3570     101       409
neutra         971     180       889
positiva       494      63      3523
Modelo guardado → models/nb_tfidf_v1.joblib
Predicción guardada → submissions/submission_nb_tfidf_v1.csv  (8500 predicciones)
Distribución: clase 0: 50.0%, clase 1: 3.0%, clase 2: 47.0%


El modelo alcanza un F1-macro de 0.5753, pero los resultados son muy disparejos entre clases. Negativa y positiva obtienen F1 de 0.78 y 0.79 respectivamente, mientras que la clase neutra colapsa a 0.15. Esto se explica por los priors aprendidos del data (40/20/40): antes de leer una sola palabra el modelo ya cree que una reseña tiene el doble de chances de ser negativa o positiva que neutra, y cuando el texto es ambiguo elige las clases mayoritarias. La matriz de confusión lo confirma, de las 2040 reseñas neutras reales, el modelo clasificó correctamente solo 180 (8.8%), derivando 971 hacia negativa y 889 hacia positiva. El siguiente experimento fuerza priors uniformes para corregir este sesgo.